In [3]:
from pyensembl import EnsemblRelease
import pandas as pd


In [4]:
#!pip install pyensembl

In [5]:

def get_genes_at_locus(chromosome, position, release=84):
    data = EnsemblRelease(release)
    return data.gene_names_at_locus(contig=chromosome, position=position)

# genes related to prostate cancer
# now: genes_file is JSON file downloaded from https://portal.gdc.cancer.gov
def rel_to_cancer(gene_name, genes_file):
    with open(genes_file) as f:
        content = f.read()
        if '\"' + gene_name + '\"' in content: 
            return True
        return False

# print(get_genes_at_locus(20, 50940000))

def read_vcf(file_name):
    with open(file_name) as f:
        for line in f:
            if line[0]=='#' and line[1]!='#':
                col_names = line[1:].split()
                break
    return pd.read_csv(file_name, sep='\t', comment='#', header=None, names = col_names, dtype={'CHROM': str})


def read_ac(file_name):
    data = pd.read_csv(file_name, sep='\t')
    data['refPos']+=1  # to compare with WES positions; it prints log 'INFO:numexpr.utils:NumExpr defaulting to 4 threads'
    return data

def filter_mutations(data_wes, data_st):  
# returns a set of tuples (chromosome, position, ref_base, alt_base) for mutations that are both in st and wes (not in germline)
    #data_wes = data_wes.loc[~data_wes['INFO'].str.contains("Germline")]  # removing germline mutations
    data_wes = data_wes.loc[data_wes['INFO'].str.contains("Somatic")]  # leaving only 'LikelySomatic' and 'StrongSomatic'
    wes_mutations = set([tuple(row) for row in data_wes[['CHROM','POS','REF','ALT']].itertuples(index=False)])
    st_mutations = set([tuple(row) for row in data_st[['refContig','refPos','refAllele','base']].itertuples(index=False)])
    return wes_mutations.intersection(st_mutations)


def filter_df(data, data_type, mutations):  # limit wes/st data to rows with specific mutations
    d = data.copy()
    d['save'] = [0]*len(d)
    for index, row in d.iterrows():
        if data_type=='wes':
            if tuple(row[['CHROM','POS','REF','ALT']]) in mutations:
                d.loc[index, 'save'] = 1
        elif data_type=='st':
            if tuple(row[['refContig','refPos','refAllele','base']]) in mutations:
                d.loc[index, 'save'] = 1
    return data.loc[d['save']==1]

def write_st(data, name):
	data.to_csv(name, sep='\t', index=False)

In [9]:
breast_genotype = pd.read_csv('breast.txt',sep='\t')
prostate_genotype = pd.read_csv('prostate.txt',sep='\t')


In [11]:
breast_genotype.columns.values

array(['clone1', 'clone2', 'clone3', 'clone4', 'clone5', 'clone6',
       'clone7'], dtype=object)

array(['chr1 1211774', 'chr1 1495699', 'chr1 8656223', 'chr1 9730666',
       'chr1 13778599', 'chr1 26188285', 'chr1 37537765', 'chr1 39562342',
       'chr1 40059381', 'chr1 45719223', 'chr1 46066115', 'chr1 53832515',
       'chr1 58782450', 'chr1 67430140', 'chr1 85270826', 'chr1 88983240',
       'chr1 93201957', 'chr1 108696298', 'chr1 108779404',
       'chr1 110363803', 'chr1 144425520', 'chr1 144427904',
       'chr1 145978445', 'chr1 148104628', 'chr1 148123960',
       'chr1 151055125', 'chr1 154273396', 'chr1 154945030',
       'chr1 155059964', 'chr1 155085402', 'chr1 155208818',
       'chr1 155261659', 'chr1 155670392', 'chr1 155916629',
       'chr1 156076894', 'chr1 173941266', 'chr1 180096175',
       'chr1 186363348', 'chr1 206454955', 'chr1 211946666',
       'chr1 212360767', 'chr1 225847137', 'chr1 226548931',
       'chr1 235135749', 'chr1 241590254', 'chr1 9037633',
       'chr1 20893548', 'chr1 26282327', 'chr1 32226471',
       'chr1 111766723', 'chr1 11528626

In [33]:
#textfile = open("breast_genes.txt", "w")
genes = []
for gene_id in breast_genotype.index.values:
    chromosome,position = gene_id.split(" ")
    #genes.merge(get_genes_at_locus(chromosome.split("chr")[1], int(position)))
    #textfile.write(get_genes_at_locus(chromosome.split("chr")[1], int(position)) + "\n")
    genes = [*genes, *get_genes_at_locus(chromosome.split("chr")[1], int(position))]
#textfile.close()

In [34]:
f=open('breast_genes.txt','w')
s1='\n'.join(genes)
f.write(s1)
f.close()

In [37]:
prostate_genotype.index.values

array(['1 11750633', '1 39028902', '1 178411772', '1 197363963',
       '1 246792089', '1 634228', '1 11750616', '1 44777740',
       '1 44777751', '1 44777755', '1 46316595', '1 50326862',
       '1 16771520', '1 22639424', '1 31652702', '1 32680511',
       '1 44778033', '1 44778047', '2 94876810', '2 130190501',
       '2 222726641', '2 130294221', '2 177219312', '2 231256445',
       '2 24334731', '2 37204496', '2 63684674', '2 113598782',
       '2 113598836', '2 113598845', '2 119368252', '2 135117509',
       '2 216194968', '3 242440', '3 39411700', '3 61742826',
       '3 96617762', '3 161429519', '3 12836140', '3 12840176',
       '3 14181597', '3 9933461', '3 12787395', '3 40457989',
       '3 47850100', '3 143518137', '3 184150208', '3 184150219',
       '4 15730972', '4 80393084', '4 108622530', '4 120342486',
       '4 151104549', '4 8619633', '4 46723939', '4 151104551',
       '4 127812779', '5 33162399', '5 33162442', '5 33162716',
       '5 77291621', '5 80650051', '5 

In [38]:
#textfile = open("breast_genes.txt", "w")
genes = []
for gene_id in prostate_genotype.index.values:
    chromosome,position = gene_id.split(" ")
    #genes.merge(get_genes_at_locus(chromosome.split("chr")[1], int(position)))
    #textfile.write(get_genes_at_locus(chromosome.split("chr")[1], int(position)) + "\n")
    genes = [*genes, *get_genes_at_locus(chromosome, int(position))]
#textfile.close()
    


In [39]:
f=open('prostate_genes.txt','w')
s1='\n'.join(genes)
f.write(s1)
f.close()

In [15]:
!pyensembl install --release 84 --species homo_sapiens

2022-07-07 06:56:37,181 - pyensembl.shell - INFO - Running 'install' for EnsemblRelease(release=84, species='homo_sapiens')
2022-07-07 06:56:37,182 - pyensembl.download_cache - INFO - Fetching /Users/shadi/Library/Caches/pyensembl/GRCh38/ensembl84/Homo_sapiens.GRCh38.84.gtf.gz from URL ftp://ftp.ensembl.org/pub/release-84/gtf/homo_sapiens/Homo_sapiens.GRCh38.84.gtf.gz
2022-07-07 06:56:37,182 - datacache.download - INFO - Downloading ftp://ftp.ensembl.org/pub/release-84/gtf/homo_sapiens/Homo_sapiens.GRCh38.84.gtf.gz to /Users/shadi/Library/Caches/pyensembl/GRCh38/ensembl84/Homo_sapiens.GRCh38.84.gtf.gz
2022-07-07 06:56:47,056 - pyensembl.download_cache - INFO - Fetching /Users/shadi/Library/Caches/pyensembl/GRCh38/ensembl84/Homo_sapiens.GRCh38.cdna.all.fa.gz from URL ftp://ftp.ensembl.org/pub/release-84/fasta/homo_sapiens/cdna/Homo_sapiens.GRCh38.cdna.all.fa.gz
2022-07-07 06:56:47,057 - datacache.download - INFO - Downloading ftp://ftp.ensembl.org/pub/release-84/fasta/homo_sapiens/cdna/

In [19]:
print(get_genes_at_locus(20, 50940000))

['ADNP-AS1', 'DPM1']
